# Cross-sector LCOX: LCOE and LCOC

This notebook compares deterministic and probabilistic levelized cost of product (**LCOX**) results using the same engines and plotting helpers as the NPV and levelized-net-margin analyses.

- Electricity is reported as **LCOE (EUR/MWh)**.
- Cement is reported as **LCOC (EUR/t cement)**.
- Lower values are economically preferable, so LCOX rankings use the lowest cost as rank 1.

The two sectors are shown separately because their products and units are different; an LCOE value should not be numerically compared with an LCOC value.

## Definition and cost boundary

For each technology and Monte Carlo draw:

$$\mathrm{LCOX}=\frac{I_0 + \sum_{t=1}^{T} C_t/(1+r)^t}{\sum_{t=1}^{T} X_t/(1+r)^t}$$

`I₀` is year-zero CAPEX, `Cₜ` contains annual fixed OPEX, variable OPEX, fuel, energy (cement electricity), and carbon cost, and `Xₜ` is annual electricity or cement output. Product sales revenue is excluded from LCOX. The lifetime and discount-rate assumptions are exactly the same as in NPV and levelized net margin (LNM). With the current constant-price and constant-output model, `product price − LCOX = LNM`.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity.electricity_npv_deterministic import calculate_deterministic_electricity_results
from electricity.electricity_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_SAMPLE_SIZE,
    simulate_electricity_results,
)
from electricity.electricity_npv_summary_figures import (
    ELECTRICITY_FINANCIAL_METRIC_OPTIONS,
    ELECTRICITY_TECHNOLOGY_LABELS,
    calculate_electricity_npv_rankings_from_results,
    electricity_npv_distribution_summary,
)
from cement.cement_npv_deterministic import calculate_deterministic_cement_results
from cement.cement_npv_monte_carlo import DEFAULT_RETROFIT_BAU_MODE, simulate_cement_results
from cement.cement_npv_summary_figures import (
    CEMENT_FINANCIAL_METRIC_OPTIONS,
    CEMENT_TECHNOLOGY_LABELS,
    calculate_cement_npv_rankings_from_results,
    cement_npv_distribution_summary,
)
from npv_summary import deterministic_metric
from npv_summary_plots import (
    fixed_financial_metric_bar_axis_config,
    plot_average_rank_bars,
    plot_financial_metric_technology_bars,
)

In [2]:
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE
FINANCIAL_METRIC = "LCOX"

ELECTRICITY_CONFIG = ELECTRICITY_FINANCIAL_METRIC_OPTIONS[FINANCIAL_METRIC]
CEMENT_CONFIG = CEMENT_FINANCIAL_METRIC_OPTIONS[FINANCIAL_METRIC]
pd.options.display.float_format = "{:,.3f}".format

## Run each model once

The same Monte Carlo arrays are reused for distribution summaries, rankings, and identity checks. This prevents subtle differences caused by rerunning simulations and avoids duplicate computation.

In [3]:
electricity_mc_results = simulate_electricity_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
)
cement_mc_results = simulate_cement_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    retrofit_bau_mode=RETROFIT_BAU_MODE,
)
electricity_deterministic_results = calculate_deterministic_electricity_results()
cement_deterministic_results = calculate_deterministic_cement_results()

In [4]:
def distribution_table(summary, metric_column):
    rows = [
        {
            "Technology": label,
            metric_column: values["mean"],
            "Median": values["median"],
            "P05": values["p05"],
            "P95": values["p95"],
        }
        for label, values in summary.items()
    ]
    return pd.DataFrame(rows).sort_values(metric_column)


def plot_lcox_distribution(summary, config, title, axis_limits, axis_ticks):
    means = {label: values["mean"] for label, values in summary.items()}
    medians = {label: values["median"] for label, values in summary.items()}
    p05 = {label: values["p05"] for label, values in summary.items()}
    p95 = {label: values["p95"] for label, values in summary.items()}
    plot_financial_metric_technology_bars(
        values=means,
        median_values=medians,
        lower_values=p05,
        upper_values=p95,
        output_path=None,
        title=title,
        sample_size=SAMPLE_SIZE,
        random_seed=RANDOM_SEED,
        x_axis_label=str(config["axis_label"]),
        x_axis_limits=axis_limits,
        x_axis_ticks=axis_ticks,
        higher_is_better=False,
        color_by_sign=False,
        zero_baseline=True,
    )
    plt.show()


def plot_deterministic_lcox(values, config, title, axis_limits, axis_ticks):
    plot_financial_metric_technology_bars(
        values=values,
        output_path=None,
        title=title,
        x_axis_label=str(config["axis_label"]),
        x_axis_limits=axis_limits,
        x_axis_ticks=axis_ticks,
        higher_is_better=False,
        color_by_sign=False,
        zero_baseline=True,
    )
    plt.show()

## Electricity: probabilistic and deterministic LCOE

In [5]:
electricity_mc_summary = electricity_npv_distribution_summary(
    electricity_mc_results,
    financial_metric=FINANCIAL_METRIC,
)
electricity_deterministic_lcoe = deterministic_metric(
    electricity_deterministic_results,
    labels=ELECTRICITY_TECHNOLOGY_LABELS,
    metric_column=str(ELECTRICITY_CONFIG["metric_column"]),
    scale=float(ELECTRICITY_CONFIG["scale"]),
)
electricity_axis_limits, electricity_axis_ticks = fixed_financial_metric_bar_axis_config(
    sector="electricity",
    financial_metric=FINANCIAL_METRIC,
    distribution_summary=electricity_mc_summary,
    deterministic_values=electricity_deterministic_lcoe,
)

display(distribution_table(electricity_mc_summary, "Mean LCOE (EUR/MWh)"))
plot_lcox_distribution(
    electricity_mc_summary,
    ELECTRICITY_CONFIG,
    "Monte Carlo mean LCOE by electricity technology",
    electricity_axis_limits,
    electricity_axis_ticks,
)

display(
    pd.DataFrame(
        electricity_deterministic_lcoe.items(),
        columns=["Technology", "Deterministic LCOE (EUR/MWh)"],
    ).sort_values("Deterministic LCOE (EUR/MWh)")
)
plot_deterministic_lcox(
    electricity_deterministic_lcoe,
    ELECTRICITY_CONFIG,
    "Deterministic LCOE by electricity technology",
    electricity_axis_limits,
    electricity_axis_ticks,
)

,Technology,Mean LCOE (EUR/MWh),Median,P05,P95
7,PV,76.580,76.579,69.031,84.068
6,Wind Onshore,80.390,80.398,69.981,90.765
5,Wind Offshore,86.817,86.775,73.598,100.096
2,CCGT,122.957,120.478,88.537,165.933
3,CCGT + CCS,131.747,129.032,90.876,182.063
4,Nuclear,143.161,143.004,96.166,190.334
0,Hard coal,158.760,158.233,147.723,171.567
1,Hard coal + CCS,163.767,163.805,138.810,189.006
8,Biogas,339.126,339.115,306.721,371.366


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42983/1324507279.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Technology,Deterministic LCOE (EUR/MWh)
7,PV,76.173
6,Wind Onshore,79.754
5,Wind Offshore,86.260
2,CCGT,122.512
3,CCGT + CCS,131.636
4,Nuclear,142.633
0,Hard coal,157.956
1,Hard coal + CCS,162.154
8,Biogas,336.610


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42983/1324507279.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Cement: probabilistic and deterministic LCOC

In [6]:
cement_mc_summary = cement_npv_distribution_summary(
    cement_mc_results,
    financial_metric=FINANCIAL_METRIC,
)
cement_deterministic_lcoc = deterministic_metric(
    cement_deterministic_results,
    labels=CEMENT_TECHNOLOGY_LABELS,
    metric_column=str(CEMENT_CONFIG["metric_column"]),
    scale=float(CEMENT_CONFIG["scale"]),
)
cement_axis_limits, cement_axis_ticks = fixed_financial_metric_bar_axis_config(
    sector="cement",
    financial_metric=FINANCIAL_METRIC,
    distribution_summary=cement_mc_summary,
    deterministic_values=cement_deterministic_lcoc,
)

display(distribution_table(cement_mc_summary, "Mean LCOC (EUR/t cement)"))
plot_lcox_distribution(
    cement_mc_summary,
    CEMENT_CONFIG,
    "Monte Carlo mean LCOC by cement technology",
    cement_axis_limits,
    cement_axis_ticks,
)

display(
    pd.DataFrame(
        cement_deterministic_lcoc.items(),
        columns=["Technology", "Deterministic LCOC (EUR/t cement)"],
    ).sort_values("Deterministic LCOC (EUR/t cement)")
)
plot_deterministic_lcox(
    cement_deterministic_lcoc,
    CEMENT_CONFIG,
    "Deterministic LCOC by cement technology",
    cement_axis_limits,
    cement_axis_ticks,
)

,Technology,Mean LCOC (EUR/t cement),Median,P05,P95
7,CCS,103.142,101.778,76.934,134.010
8,Process heat integration,105.438,105.457,98.209,112.646
6,Waste heat recovery,105.858,105.774,99.986,112.074
3,Clinker substitution,105.990,106.008,98.321,113.603
4,Alternative fuels,106.099,106.102,97.886,114.287
5,Efficiency improvement,107.971,107.927,101.381,114.760
0,BAU,109.161,109.173,102.312,115.984
1,Electrification,254.552,256.913,193.549,307.802
2,Electrolysis,532.488,523.016,352.212,740.055


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42983/1324507279.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Technology,Deterministic LCOC (EUR/t cement)
7,CCS,100.396
8,Process heat integration,102.139
6,Waste heat recovery,102.620
4,Alternative fuels,102.626
3,Clinker substitution,102.874
5,Efficiency improvement,104.564
0,BAU,105.572
1,Electrification,251.925
2,Electrolysis,526.607


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42983/1324507279.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
for sector, results, labels, ranking_function, metric_label in (
    (
        "Electricity",
        electricity_mc_results,
        ELECTRICITY_TECHNOLOGY_LABELS,
        calculate_electricity_npv_rankings_from_results,
        "LCOE",
    ),
    (
        "Cement",
        cement_mc_results,
        CEMENT_TECHNOLOGY_LABELS,
        calculate_cement_npv_rankings_from_results,
        "LCOC",
    ),
):
    _, ranking_summary = ranking_function(
        results=results,
        sector_name=sector,
        financial_metric=FINANCIAL_METRIC,
    )
    ranking_summary = ranking_summary.assign(
        display_label=ranking_summary["technology"].map(labels).fillna(ranking_summary["technology"])
    )
    display(
        ranking_summary.loc[
            :, ["display_label", "average_rank", "probability_rank_1", "probability_top_3"]
        ].rename(
            columns={
                "display_label": "Technology",
                "average_rank": "Average rank",
                "probability_rank_1": "Probability rank 1",
                "probability_top_3": "Probability top 3",
            }
        )
    )
    plot_average_rank_bars(
        ranking_summary=ranking_summary,
        output_path=None,
        title=f"Monte Carlo {metric_label} ranking — {sector}",
        metric_label=metric_label,
        random_seed=RANDOM_SEED,
        higher_is_better=False,
    )
    plt.show()

,Technology,Average rank,Probability rank 1,Probability top 3
0,PV,1.513,0.579,0.998
1,Wind Onshore,1.987,0.303,0.983
2,Wind Offshore,2.664,0.115,0.919
3,CCGT,4.534,0.001,0.052
4,CCGT + CCS,5.546,0.002,0.031
5,Nuclear,5.932,0.000,0.017
6,Hard coal,6.764,0.000,0.000
7,Hard coal + CCS,7.058,0.000,0.000
8,Biogas,9.000,0.000,0.000


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42983/1211903033.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Technology,Average rank,Probability rank 1,Probability top 3
0,Process heat integration,2.992,0.128,0.661
1,Waste heat recovery,3.229,0.088,0.582
2,CCS,3.487,0.521,0.578
3,Clinker substitution,3.515,0.118,0.535
4,Alternative fuels,3.634,0.141,0.525
5,Efficiency improvement,4.998,0.004,0.119
6,BAU,6.145,0.000,0.002
7,Electrification,8.000,0.000,0.000
8,Electrolysis,9.000,0.000,0.000


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42983/1211903033.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Internal consistency check

In [8]:
def maximum_identity_error(results, price_column, lcox_column, lnm_column):
    return max(
        float(
            np.max(
                np.abs(
                    np.asarray(values[price_column], dtype=float)
                    - np.asarray(values[lcox_column], dtype=float)
                    - np.asarray(values[lnm_column], dtype=float)
                )
            )
        )
        for values in results.values()
    )

identity_check = pd.DataFrame(
    {
        "Sector": ["Electricity", "Cement"],
        "Maximum |price - LCOX - LNM|": [
            maximum_identity_error(
                electricity_mc_results,
                "electricity_price_eur_per_mwh",
                "lcoe_eur_per_mwh",
                "levelized_net_margin_eur_per_mwh",
            ),
            maximum_identity_error(
                cement_mc_results,
                "cement_price_eur_per_t",
                "lcoc_eur_per_t",
                "levelized_net_margin_eur_per_t",
            ),
        ],
    }
)
display(identity_check)

,Sector,Maximum |price - LCOX - LNM|
0,Electricity,0.000
1,Cement,0.000


## Reproducible file outputs

To regenerate dated figures plus raw and processed CSV files with the shared production pipeline, use:

```bash
PYTHONPATH=src /opt/anaconda3/envs/master-thesis/bin/python -m electricity.electricity_npv_summary_figures --kind all --ranking-output both --metric LCOX
PYTHONPATH=src /opt/anaconda3/envs/master-thesis/bin/python -m cement.cement_npv_summary_figures --kind all --ranking-output both --metric LCOX
```

These commands use the same calculations and plot helpers as this notebook.